<a href="https://colab.research.google.com/github/revc-1993/script-conciliacion-cz8s/blob/main/script_conciliacion_cz8s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Script para identificar saldos por conciliar CZ8S**

### **1. Carga de datos**

In [1]:
# Conectar google
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# PATH de archivos CSV
csv_mayor_contable = '/content/drive/MyDrive/Trabajo/CZ8-S/ADMIN/mayores_contables.csv'
csv_kardex = '/content/drive/MyDrive/Trabajo/CZ8-S/ADMIN/kardex.csv'

In [ ]:
import pandas as pd
df_mayores = pd.read_csv(csv_mayor_contable, delimiter=';', decimal=',')
df_kardex  = pd.read_csv(csv_kardex, delimiter=';', decimal=',')

### **2. Extraer Cod. Ingreso a Bodega**

In [ ]:
import re
def extraer_codigo(text):
    match = re.search(r'IB-\d+', text)
    return match.group(0) if match else None

df_mayores['cod_bodega'] = df_mayores['descripcion'].apply(extraer_codigo)

### **3. Identificar ingresos contables sin ajustes**

In [ ]:
# Ingresos y ajustes
ingresos = df_mayores[df_mayores['tipo_asiento'] == 'INGRESO']
ajustes = df_mayores[df_mayores['tipo_asiento'] == 'AJUSTE']

# Cruce de información con merge() usando LEFT JOIN, buscando que haya en
# cada ajuste, un cod_bodega que se relacione con el ingreso en cuestión
auditoria = pd.merge(
    ingresos,
    ajustes[['cod_bodega', 'haber']],
    on='cod_bodega',
    how='left',
    suffixes=('_ing', '_aju')
)

# Separo los ingresos contables que no tienen nada en haber_aju
no_conciliados = auditoria[auditoria['haber_aju'].isna()].copy()

# Busco los ingresos no conciliados en el Kardex
resultado_final = pd.merge(
    no_conciliados,
    df_kardex,
    left_on='cod_bodega',
    right_on='COD_MOVIMIENTO',
    how='inner'
)

# Busco los items por cod_producto a filtrar del kardex
items_conciliar = resultado_final['cod_producto'].unique()

# Filtro todos los registros relacionados con los items de conciliar
items_egresos = df_kardex[df_kardex['id_producto'].isin(items_conciliar)]

# Filtro los egresos de items_egresos, y obtengo los egresos a buscar
pendientes_busqueda = items_egresos[items_egresos['tipo_movimiento'] == 'EGRESO']

pendientes_busqueda